In [ ]:
%sql
DROP TABLE IF EXISTS workspace.gold_weather.fact_weather_daily;

CREATE TABLE IF NOT EXISTS workspace.gold_weather.fact_weather_daily (
  location_id BIGINT,
  date_key BIGINT,
  data_type STRING,
  weather_code BIGINT,
  temp_max DOUBLE,
  temp_min DOUBLE,
  temp_mean DOUBLE,
  precipitation_sum DOUBLE,
  wind_speed_max DOUBLE,
  wind_gusts_max DOUBLE,
  shortwave_radiation_sum DOUBLE,
  et0_fao_evapotranspiration DOUBLE,
  uv_index_max DOUBLE,
  sunshine_duration DOUBLE
)

In [ ]:
# El bloque "daily" de Open-Meteo ya viene agregado por dia local
# (timezone=auto); fact_weather_daily solo tipa/renombra, no vuelve a agregar
# desde weather_hourly (ver Proyecto/DECISIONS.md #10)
fact = spark.sql("""
    SELECT
        location_id,
        CAST(date_format(observation_date, 'yyyyMMdd') AS BIGINT) AS date_key,
        data_type,
        weather_code,
        temperature_2m_max AS temp_max,
        temperature_2m_min AS temp_min,
        temperature_2m_mean AS temp_mean,
        precipitation_sum,
        wind_speed_10m_max AS wind_speed_max,
        wind_gusts_10m_max AS wind_gusts_max,
        shortwave_radiation_sum,
        et0_fao_evapotranspiration,
        uv_index_max,
        sunshine_duration
    FROM workspace.silver_weather.weather_daily
""")

fact.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_weather.fact_weather_daily")